# Pseudonymization and Cryptographic Protection of Microdata

Double-hash pseudonymization (RIPEMD-160 + SHA-512), brute-force and dictionary attacks against it, a salted fix, encryption of a sensitive attribute with RSA, ElGamal and A5/1, and an independence (fairness) check on a predicted attribute.

Scenario: the same HR datasets as in notebook 01 (`workers.csv`, `absenteeism.csv`); this time identifiers are pseudonymized instead of removed, and sensitive values are encrypted.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import ast
import pandas as pd
path ="/content/drive/MyDrive/practica_2_privacidad_seguridad_Datos/"
df_workers = pd.read_csv(path+"data/workers.csv", sep = ';')
df_workers.info()
df_workers_original = df_workers.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            22 non-null     int64 
 1   name          22 non-null     object
 2   sex           22 non-null     object
 3   age           22 non-null     int64 
 4   zip_code      22 non-null     int64 
 5   num_absences  22 non-null     int64 
 6   absences      22 non-null     object
 7   insurance     22 non-null     int64 
dtypes: int64(5), object(3)
memory usage: 1.5+ KB


In [3]:
df_absenteeism = pd.read_csv(path+"data/absenteeism.csv", sep = ';')
df_absenteeism.info()
df_absenteeism_original = df_absenteeism.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 21 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ID                               470 non-null    float64
 1   Reason_for_absence               471 non-null    int64  
 2   Month_of_absence                 470 non-null    float64
 3   Day_of_the_week                  470 non-null    float64
 4   Seasons                          470 non-null    float64
 5   Transportation_expense           470 non-null    float64
 6   Distance_from_Residence_to_Work  470 non-null    float64
 7   Service_time                     470 non-null    float64
 8   Age                              470 non-null    float64
 9   Work_load_Average_day            470 non-null    float64
 10  Hit_target                       470 non-null    float64
 11  Disciplinary_failure             470 non-null    float64
 12  Education             

**Exercise 1:** Pseudonymize the identifier attributes by replacing each value with its RIPEMD-160 hash and then hashing that result with SHA-512.

**1.1** — Identify which attributes of `df_workers` act as identifiers.

In [ ]:
# Respuesta
#id: Este es un identificador único para cada registro en la tabla.
#name: El nombre de la persona es un identificador directo.
#zip_code: El código postal puede ser usado para identificar a una persona si se combina con otros atributos.

**1.2** — Implement a RIPEMD-160 hex-digest function.

In [5]:
!pip install pycryptodome

In [6]:
from Crypto.Hash import RIPEMD160           # Importa la librería RIPEMD160 para calcular el hash RIPEMD-160
import hashlib                              # Importa la librería hashlib para calcular hashes con varios algoritmos

In [7]:
def ripemd160_hash(text):                   # Define una función para calcular el hash RIPEMD-160 de un texto
    h = RIPEMD160.new()                     # Crea un nuevo objeto hash RIPEMD-160
    h.update(text.encode('utf-8'))          # Actualiza el objeto hash con el texto codificado en UTF-8
    return h.hexdigest()                    # Devuelve el hash calculado en formato hexadecimal

**1.3** — Chain a SHA-512 hex digest on top of the RIPEMD-160 output.

In [8]:
def sha512_hash(text):                      # Define una función para calcular el hash SHA-512 de un texto
    h = hashlib.sha512()                    # Crea un nuevo objeto hash SHA-512
    h.update(text.encode('utf-8'))          # Actualiza el objeto hash con el texto codificado en UTF-8
    return h.hexdigest()                    # Devuelve el hash calculado en formato hexadecimal

**1.4** — Combine both hashes into a single `double_hash` function.

In [9]:
# Respuesta

def double_hash(text):                      # Define una función para aplicar dos hashes consecutivos a un texto
    ripemd160_result = ripemd160_hash(text) # Calcula el hash RIPEMD-160 del texto
    sha512_result = sha512_hash(ripemd160_result) # Calcula el hash SHA-512 del resultado anterior
    return sha512_result                    # Devuelve el resultado final del doble hash

**1.5** — Apply the pseudonymization to the dataframe and discuss why the process is needed.

In [10]:
# Respuesta

# Pseudonimizamos los identificadores
def pseudonymize_dataframe(df):             # Define una función para pseudonimizar identificadores en un DataFrame
    df['id'] = df['id'].astype(str).apply(double_hash)   # Aplica el doble hash a la columna 'id' después de convertirla a string
    df['name'] = df['name'].apply(double_hash)           # Aplica el doble hash a la columna 'name'
    df['zip_code'] = df['zip_code'].astype(str).apply(double_hash) # Aplica el doble hash a la columna 'zip_code' después de convertirla a string
    return df                                 # Devuelve el DataFrame con los identificadores pseudonimizados

# Aplicamos pseudonimización
pseudonymized_workers_df = pseudonymize_dataframe(df_workers_original) # Llama a la función para pseudonimizar el DataFrame original
print(pseudonymized_workers_df)            # Imprime el DataFrame pseudonimizado

                                                   id  \
0   ceb51fb0fd5af1a95fc834731e16fe8fb8ad1ae4457042...   
1   06323d7afcbefa79a1e62f6bce86190dc83c11ed7334ac...   
2   a68d955d2c7dade51bd2c94385b9e772bbb939e36f95b3...   
3   81b645473612de18789ab375b11a27193b4c238ec361ce...   
4   28a46e4cef69414806eeb28d32c477f2ef1ecbe4cb1fb6...   
5   6bdf137e846390a3c5b58ed6962aa95a21afc4b240fc05...   
6   ad1a065aa70eeb69ccccf9170d06570f389435bf2b13e5...   
7   b2578d881e20c187b8df9d8e98a58041035a282a4eb8fe...   
8   98204a99b0b13f66ec431754ecd7fab07186e1ea6fde45...   
9   70a7a9b92c74803050a9f46a68ae929aed01ec25b3f036...   
10  02825ce71d3182e9c55fd0204f38a5075190cbfe538dc6...   
11  46d1cb3a25b4ea17fd877d08706a7efdddb2507655cd0e...   
12  27110cad9c225334272d8bcd0d725aa059db2bdb1a16f6...   
13  0a827fefb08c218cc0609b4f3202a3a6952c5538b9156b...   
14  0d315b61de345592dd9e6df818909d613a9779a26a9d24...   
15  becdb3e0dab4eb7b0dc657473fc8c7194bebe711e11ba0...   
16  f360f62f443536a5d8015317f76

<div class="alert alert-block alert-success">
Justificación:

La pseudonimización es una técnica esencial en la protección de datos personales, especialmente en contextos donde se maneja información sensible, como en el departamento de Recursos Humanos de la administración pública. El proceso de pseudonimización implica sustituir identificadores directos por valores derivados mediante funciones de hash. Esto tiene varias ventajas:

- Al transformar los identificadores en valores hash, se previene la identificación directa de los individuos, protegiendo así su privacidad.
En muchos países, la legislación sobre protección de datos (como el GDPR en Europa) exige la anonimización o pseudonimización de datos personales para minimizar riesgos en caso de violaciones de datos.

- En caso de acceso no autorizado a los datos, la información pseudonimizada no revelará fácilmente la identidad de los individuos, reduciendo el riesgo de uso indebido.

Sabiendo esto:

Se identificaron los atributos id, name y zip_code como identificadores directos que necesitan ser pseudonimizados.
Se utilizó una combinación de RIPEMD-160 y SHA-512 para obtener valores hash robustos. RIPEMD-160 proporciona una buena mezcla inicial y SHA-512 asegura una alta resistencia contra ataques de colisión.
Los valores hash se aplicaron a los identificadores seleccionados, reemplazando los datos originales con sus valores hash correspondientes.


Aunque el proceso actual utiliza dos funciones de hash para maximizar la seguridad, hay formas de optimizar el proceso:

- Usar solo SHA-512 puede ser suficientemente seguro y más eficiente. SHA-512 es un algoritmo robusto por sí mismo y elimina la necesidad de aplicar dos funciones de hash.

- Para grandes volúmenes de datos, se puede aplicar procesamiento en paralelo o en lotes para mejorar la eficiencia. Usar bibliotecas como multiprocessing en Python puede acelerar significativamente el proceso.

- Si se espera que ciertos identificadores se repitan, el cacheo de resultados intermedios puede evitar cálculos redundantes, mejorando la eficiencia.



</div>

**Exercise 2:** Attack the pseudonymized dataset knowing one person's full name — **2.1** without knowing which hash functions were used, and **2.2** knowing them — timing both attempts.

In [11]:
nombreConocido = "Honora Garbutt"

In [12]:
import itertools                                  # Importa la librería itertools para manejar iteraciones
import time                                       # Importa la librería time para medir el tiempo de ejecución

# Inicio del tiempo de ejecución
start_time = time.time()                          # Guarda el tiempo actual para medir el tiempo de ejecución

# Simulación de un ataque de fuerza bruta
# NOTA: Este método no será exitoso sin el conocimiento del hash y es solo un ejemplo teórico
for id_hash, name_hash, age, zip_code, absences in zip(    # Itera sobre los valores originales del DataFrame
        df_workers_original['id'], df_workers_original['name'], df_workers_original['age'],
        df_workers_original['zip_code'], df_workers_original['absences']):

    # Esto es simplemente una comprobación; en realidad, necesitamos descifrar el hash,
    # lo cual no es factible sin el conocimiento del algoritmo de hash
    if nombreConocido in name_hash:                # Comprueba si el nombre conocido está en el hash del nombre
        print(f"ID Hash: {id_hash}")               # Imprime el hash del ID
        print(f"Name Hash: {name_hash}")           # Imprime el hash del nombre
        print(f"Age: {age}")                       # Imprime la edad
        print(f"Zip Code Hash: {zip_code}")        # Imprime el hash del código postal
        print(f"Absences: {absences}")             # Imprime las ausencias

# Fin del tiempo de ejecución
end_time = time.time()                             # Guarda el tiempo actual al finalizar la ejecución
execution_time = end_time - start_time             # Calcula el tiempo de ejecución restando el tiempo de inicio

print(f"Execution time: {execution_time} seconds") # Imprime el tiempo de ejecución


Execution time: 0.0008578300476074219 seconds


In [13]:
# Respuesta
# Inicio del tiempo de ejecución

start_time = time.time()                          # Guarda el tiempo actual para medir el tiempo de ejecución

# Generamos el hash doble del nombre conocido
nombre_conocido_hash = double_hash(nombreConocido) # Calcula el hash doble del nombre conocido

# Buscamos el hash en el DataFrame pseudonimizado
matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == nombre_conocido_hash] # Busca el hash en el DataFrame pseudonimizado

result_2_2 = None                                 # Inicializa la variable para almacenar el resultado

# Si encontramos una coincidencia, mostramos la información
if not matched_row.empty:                         # Verifica si se encontró una coincidencia
    result_2_2 = matched_row.to_dict('records')[0]# Convierte la fila coincidente en un diccionario y lo almacena en result_2_2

# Fin del tiempo de ejecución
end_time = time.time()                            # Guarda el tiempo actual al finalizar la ejecución
execution_time_2_2 = end_time - start_time        # Calcula el tiempo de ejecución restando el tiempo de inicio

result_2_2, execution_time_2_2                    # Muestra el resultado y el tiempo de ejecución

({'id': '0d315b61de345592dd9e6df818909d613a9779a26a9d242b41076222996c70945ce127fd829f0ad1a4c1aa3158f64f73fde1027b0b7b49309ee47d9419dc3dcd',
  'name': '2309b71c489923706bd137119e2ec2c589e14a22d942a88cbe913c282d3fc2cf339dae75180174d0aeb24f69f5ab138fe59a789dd28af06b033680848aa7b6de',
  'sex': 'F',
  'age': 43,
  'zip_code': '458aba1e7615af32248bd800e2ef90198b4e52dba4be7b2ca2bf1f8fd2015be70d299c30d32a49e88aa4e26dbe248e2dfedb2e53cd25b71a15ac072a73b0ab46',
  'num_absences': 37,
  'absences': '[5-9-241476,2-9-241476,5-10-253465,3-10-253465,5-11-306345,2-11-306345,5-11-306345,4-3-343253,4-4-326452,6-4-326452,6-4-326452,5-6-377550,3-9-294217,6-9-294217,6-9-294217,2-9-294217,3-10-265017,3-10-265017,2-11-284031,6-11-284031,4-1-330061,3-2-251818,2-6-253957,5-7-230290,5-8-249797,5-8-249797,5-9-261756,4-10-284853,6-10-284853,4-10-284853,4-11-268519,3-11-268519,4-12-280549,5-5-237656,4-5-237656,5-5-237656,5-6-275089]',
  'insurance': 1},
 0.012212038040161133)

<div class="alert alert-block alert-success">
Justificación:


En este caso, intentamos recuperar información sin conocer las funciones hash utilizadas. Este enfoque es casi imposible de tener éxito debido a la naturaleza unidireccional de los algoritmos de hash. Los algoritmos de hash están diseñados para que no sea factible revertir el hash a su valor original sin conocer la información específica del hash, lo que proporciona una capa de seguridad robusta para los datos pseudonimizados.

Por otro lado, sabiendo las funciones hash utilizadas, podemos generar el hash correspondiente a un nombre conocido y compararlo con los valores en el DataFrame pseudonimizado. Este enfoque demuestra que la seguridad de los datos pseudonimizados depende en gran medida de mantener la confidencialidad sobre los algoritmos de hash utilizados.

Para el proceso de recuperación de información, se utilizó el algoritmo RIPEMD-160 seguido de SHA-512 para generar el hash del nombre conocido "Honora Garbutt".
Se buscó el hash generado en el DataFrame pseudonimizado para encontrar una coincidencia.
Si se encontró una coincidencia, se recuperaron los datos asociados.

Sin conocer las funciones Hash no se pudo recuperar ninguna información debido a la imposibilidad de revertir el hash sin conocer el algoritmo utilizado.

En este ejercicio podemos observar la importancia de mantener la confidencialidad de los algoritmos de hash utilizados en la pseudonimización de datos. Aunque los datos están protegidos mediante hashing, el conocimiento del algoritmo de hash puede permitir a un atacante recuperar información sensible. Por lo tanto, es crucial implementar políticas y controles que aseguren la confidencialidad de estos métodos para mantener la seguridad y privacidad de los datos.










</div>



**Exercise 3:** Dictionary attack — the attacker only knows the workers' country and builds candidate names from lists of common first and last names.

In [14]:
lista_nombres_comunes = ["Daisy", "Philip", "Wilfred", "Honora", "Gale", "Jack", "Roland", "Mary"]
lista_apellidos_comunes = ["Garry", "Neal", "Spooner", "Wade", "Home", "Garbutt", "Brigham", "Wite"]

**3.1** — Knowing the hash functions, recover a person's age, zip code and absences; report the cost in number of hashes computed.

In [15]:
# Respuesta

# Generamos todos los nombres completos posibles
nombres_completos = [f"{nombre} {apellido}" for nombre, apellido in itertools.product(lista_nombres_comunes, lista_apellidos_comunes)]  # Combina todos los nombres y apellidos comunes en una lista de nombres completos

# Número de hashes calculados
num_hashes_calculados = 0  # Inicializa el contador de hashes calculados

# Búsqueda de coincidencias en el DataFrame pseudonimizado
for nombre_completo in nombres_completos:  # Itera sobre todos los nombres completos generados
    hashed_name = double_hash(nombre_completo)  # Calcula el hash doble del nombre completo
    num_hashes_calculados += 1  # Incrementa el contador de hashes calculados
    matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name]  # Busca el hash en el DataFrame pseudonimizado
    if not matched_row.empty:  # Si se encuentra una coincidencia
        print("Información recuperada:", matched_row[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Imprime la información recuperada (edad, código postal, número de ausencias)

print("Número total de hashes calculados:", num_hashes_calculados)  # Imprime el número total de hashes calculados

Información recuperada: [{'age': 58, 'zip_code': '33c896689a7f1bb8207e12fa3eeb8698cf986d7b681548ef17740536f14a1c8c008902e86b1448da40ba2969c5fea02db1040176aef0a11740bd7113db87aff6', 'num_absences': 46}]
Información recuperada: [{'age': 43, 'zip_code': '458aba1e7615af32248bd800e2ef90198b4e52dba4be7b2ca2bf1f8fd2015be70d299c30d32a49e88aa4e26dbe248e2dfedb2e53cd25b71a15ac072a73b0ab46', 'num_absences': 37}]
Información recuperada: [{'age': 50, 'zip_code': '6c760336060d569a298f56111c3014551c70d239b7d96ed062fa66151b9aad6d9af27dca0d234d5a98f31f495fdd74c2c415b0cba5e965b1b106e0d28623e344', 'num_absences': 42}]
Número total de hashes calculados: 64


<div class="alert alert-block alert-success">
Justificación:

Este método es efectivo porque combinamos conocimientos, en cuanto a las funciones hash y los nombres comunes, para realizar un ataque de diccionario. La eficacia depende de la calidad de la lista de nombres y la exactitud de la función hash conocida. Si bien este método puede ser costoso en términos de cómputo, si la lista es grande, es una forma viable de intentar desanonimizar información si se conoce el algoritmo de hash.



</div>

**3.2** — Same attack when unsure whether RIPEMD160+SHA512 or MD5+SHA256 was used; report the cost.

In [16]:
# Respuesta

def md5_hash(text):  # Define una función para calcular el hash MD5 de un texto
    h = hashlib.md5()  # Crea un nuevo objeto hash MD5
    h.update(text.encode('utf-8'))  # Actualiza el objeto hash con el texto codificado en UTF-8
    return h.hexdigest()  # Devuelve el hash calculado en formato hexadecimal

def sha256_hash(text):  # Define una función para calcular el hash SHA-256 de un texto
    h = hashlib.sha256()  # Crea un nuevo objeto hash SHA-256
    h.update(text.encode('utf-8'))  # Actualiza el objeto hash con el texto codificado en UTF-8
    return h.hexdigest()  # Devuelve el hash calculado en formato hexadecimal

def double_hash_md5_sha256(text):  # Define una función para aplicar dos hashes consecutivos (MD5 y SHA-256) a un texto
    md5_result = md5_hash(text)  # Calcula el hash MD5 del texto
    sha256_result = sha256_hash(md5_result)  # Calcula el hash SHA-256 del resultado anterior
    return sha256_result  # Devuelve el resultado final del doble hash

# Número de hashes calculados
num_hashes_calculados = 0  # Inicializa el contador de hashes calculados

# Búsqueda de coincidencias en el DataFrame pseudonimizado para ambos algoritmos
for nombre_completo in nombres_completos:  # Itera sobre todos los nombres completos generados
    hashed_name_ripe_sha = double_hash(nombre_completo)  # Calcula el hash doble (RIPEMD-160 + SHA-512) del nombre completo
    hashed_name_md5_sha256 = double_hash_md5_sha256(nombre_completo)  # Calcula el hash doble (MD5 + SHA-256) del nombre completo
    num_hashes_calculados += 2  # Cada nombre se hashea dos veces, incrementa el contador en 2
    matched_row_ripe_sha = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name_ripe_sha]  # Busca el hash (RIPEMD-160 + SHA-512) en el DataFrame pseudonimizado
    matched_row_md5_sha256 = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name_md5_sha256]  # Busca el hash (MD5 + SHA-256) en el DataFrame pseudonimizado

    if not matched_row_ripe_sha.empty:  # Si se encuentra una coincidencia con el hash (RIPEMD-160 + SHA-512)
        print("Información recuperada con RIPEMD160 + SHA512:", matched_row_ripe_sha[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Imprime la información recuperada
    if not matched_row_md5_sha256.empty:  # Si se encuentra una coincidencia con el hash (MD5 + SHA-256)
        print("Información recuperada con MD5 + SHA256:", matched_row_md5_sha256[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Imprime la información recuperada

print("Número total de hashes calculados:", num_hashes_calculados)  # Imprime el número total de hashes calculados

Información recuperada con RIPEMD160 + SHA512: [{'age': 58, 'zip_code': '33c896689a7f1bb8207e12fa3eeb8698cf986d7b681548ef17740536f14a1c8c008902e86b1448da40ba2969c5fea02db1040176aef0a11740bd7113db87aff6', 'num_absences': 46}]
Información recuperada con RIPEMD160 + SHA512: [{'age': 43, 'zip_code': '458aba1e7615af32248bd800e2ef90198b4e52dba4be7b2ca2bf1f8fd2015be70d299c30d32a49e88aa4e26dbe248e2dfedb2e53cd25b71a15ac072a73b0ab46', 'num_absences': 37}]
Información recuperada con RIPEMD160 + SHA512: [{'age': 50, 'zip_code': '6c760336060d569a298f56111c3014551c70d239b7d96ed062fa66151b9aad6d9af27dca0d234d5a98f31f495fdd74c2c415b0cba5e965b1b106e0d28623e344', 'num_absences': 42}]
Número total de hashes calculados: 128


<div class="alert alert-block alert-success">
Justificación:


Si dudamos entre dos combinaciones de algoritmos de hash (RIPEMD-160 + SHA-512 o MD5 + SHA-256), podemos implementar ambos y buscar coincidencias en el DataFrame. Este enfoque duplica el número de hashes calculados, pero sigue siendo viable. El coste del ataque es igual al número de combinaciones de nombres y apellidos multiplicado por dos (para cubrir ambas combinaciones de algoritmos de hash).


</div>

**Exercise 4:** Propose and implement a scheme (per-record salts) that defeats both previous attacks, and demonstrate that it does.

In [17]:
# Respuesta
import os

# Funciones de hash con salt
def generate_salt():                         # Genera un salt único
    return os.urandom(16).hex()              # Devuelve el salt en formato hexadecimal

def hash_with_salt(text, salt):              # Aplica el hash a un texto combinado con un salt
    combined = text + salt                   # Combina el texto original con el salt
    ripemd160_result = ripemd160_hash(combined)  # Calcula el hash RIPEMD-160 del texto combinado
    sha512_result = sha512_hash(ripemd160_result) # Calcula el hash SHA-512 del resultado anterior
    return sha512_result                     # Devuelve el resultado final del doble hash

# Funciones de hash originales
def ripemd160_hash(text):                    # Define una función para calcular el hash RIPEMD-160 de un texto
    h = RIPEMD160.new()                      # Crea un nuevo objeto hash RIPEMD-160
    h.update(text.encode('utf-8'))           # Actualiza el objeto hash con el texto codificado en UTF-8
    return h.hexdigest()                     # Devuelve el hash calculado en formato hexadecimal

def sha512_hash(text):                       # Define una función para calcular el hash SHA-512 de un texto
    h = hashlib.sha512()                     # Crea un nuevo objeto hash SHA-512
    h.update(text.encode('utf-8'))           # Actualiza el objeto hash con el texto codificado en UTF-8
    return h.hexdigest()                     # Devuelve el hash calculado en formato hexadecimal

# Pseudonimización con salt
def pseudonymize_with_salt(df):              # Define una función para pseudonimizar identificadores con salt
    salts = []                               # Inicializa una lista para almacenar los salts
    for i, row in df.iterrows():             # Itera sobre cada fila del DataFrame
        salt = generate_salt()               # Genera un salt único para cada fila
        salts.append(salt)                   # Almacena el salt en la lista
        df.at[i, 'id'] = hash_with_salt(str(row['id']), salt)  # Aplica el hash con salt al ID
        df.at[i, 'name'] = hash_with_salt(row['name'], salt)  # Aplica el hash con salt al nombre
        df.at[i, 'zip_code'] = hash_with_salt(str(row['zip_code']), salt)  # Aplica el hash con salt al código postal
    df['salt'] = salts                       # Añade la columna de salts al DataFrame
    return df                                # Devuelve el DataFrame pseudonimizado

# Aplicamos pseudonimización con salt
pseudonymized_workers_df = pseudonymize_with_salt(df_workers_original)  # Aplica la pseudonimización con salt al DataFrame original

# Guardamos el DataFrame con los hashes y salts
pseudonymized_workers_df.head()              # Muestra las primeras filas del DataFrame pseudonimizado


,id,name,sex,age,zip_code,num_absences,absences,insurance,salt
0,d7ec9551ece573219b015b1965721dcf4f5085e08fa9e2...,c05039392e0ba7e947afb0d0fd5b7492804059c2e6f855...,M,27,0f324e09a7cd4dc685fd65378a689e1330150818bace83...,23,"[2-7-239554,5-8-205917,4-12-261306,6-4-326452,...",1,aa9f47c78c153779f2ff0551fbee2263
1,04a029f3d1d9710374de57647891bd67d30ea2db9250b6...,732280a2961b2fa5b7226b06043cd49f8912eec02c87fc...,M,28,e6291f968fc370416c6bf9d46f9812ce5ad0db06786124...,6,"[5-8-205917,2-8-205917,6-8-205917,2-4-326452,2...",1,a51ade9e69d95041e1de67f97a580285
2,c6f53a43ee6e7bac45824625403b03bdc83e3902f4c1ab...,f3bfb4d22762d6f12578578573c0d3b3969f41f61c4c4c...,M,29,55d8efe63b5cd557720068e8be97588ac2d3bfaa8fff21...,113,"[4-7-239554,6-7-239554,4-7-239554,4-7-239554,6...",0,1a3cd923f45b77df9d1acaffaed4a544
3,28de6ff98a8314aebefb806aef1c8b3f5a63067079ddbd...,eb1b61dc231697589fa73edf9904c389ebb198a60f7bce...,M,30,dade35875d6b83ccb9a433fc3239019d6750579870b0e6...,1,[3-0-271219],0,4083519e2a4c10ee7ca9a753d92c53a3
4,4fc3e1a59e9b1321e91304311050ae94337836e91eb3b2...,69823e8ae1bee1a4c35ea8031821cc9db4d5b01d17ed23...,F,31,4a2ac7f279e6ce943ad425633aea058e0f0c4a1c814a78...,19,"[3-9-241476,5-9-294217,4-9-294217,5-9-294217,4...",0,2e790b72c5988b8a3e7ef73aba4e8d09


In [18]:
# Comprobación ataque 1
# Intentar ataque de diccionario sin conocer el salt
for nombre_completo in lista_nombres_comunes:         # Itera sobre la lista de nombres comunes
    hashed_name = double_hash(nombre_completo)        # Calcula el hash doble del nombre
    matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name]  # Busca el hash en el DataFrame pseudonimizado
    if not matched_row.empty:                # Si se encuentra una coincidencia
        print("Información recuperada:", matched_row[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Imprime la información recuperada

print("Ataque sin conocer el salt no tuvo éxito.")  # Mensaje indicando que el ataque no tuvo éxito

Ataque sin conocer el salt no tuvo éxito.


In [19]:
# Comprobación ataque 2
# Intentamos ataque de diccionario sin conocer los salts
for nombre_completo in nombres_completos:    # Itera sobre todos los nombres completos generados
    hashed_name = double_hash_md5_sha256(nombre_completo)  # Calcula el hash doble del nombre completo
    matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name]  # Busca el hash en el DataFrame pseudonimizado
    if not matched_row.empty:                # Si se encuentra una coincidencia
        print("Información recuperada:", matched_row[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Imprime la información recuperada

print("Ataque sin conocer los salts no tuvo éxito.")  # Mensaje indicando que el ataque no tuvo éxito

Ataque sin conocer los salts no tuvo éxito.


<div class="alert alert-block alert-success">
Justificación:

Al añadir un salt único a cada entrada, se asegura que incluso si dos entradas tienen el mismo valor, sus hashes serán diferentes.

La combinación del salt con los valores antes del hash incrementa la entropía, haciendo que los ataques de diccionario y fuerza bruta sean mucho menos efectivos.

Incluso si un atacante conoce los algoritmos de hash utilizados, sin los salts, no podrá generar los mismos hashes que se almacenan en el DataFrame.

Por tanto, esta solución demuestra que la incorporación de salts en el proceso de hashing proporciona una capa adicional de seguridad, protegiendo eficazmente contra ataques de diccionario y fuerza bruta.

</div>

**Exercise 5:** Encrypt the `absences` attribute with three different cryptosystems and compare the properties of each solution.

**5.1** — Encrypt the absences with RSA using the given public key (`df_workers_rsa`).

In [20]:
public_key_rsa = [15764585266, 15648952315]

In [21]:
# Respuesta
n, e = public_key_rsa                                      # Asignamos los valores de la clave pública RSA (n, e)

def rsa_encrypt(message, n, e):                            # Definimos la función para cifrar un mensaje usando RSA
    """Cifra un mensaje usando RSA"""
    return pow(message, e, n)                              # Aplicamos el cifrado RSA: mensaje^e mod n

# Creamos copia del dataframe para cifrado RSA
df_workers_rsa = df_workers.copy()                         # Creamos una copia del dataframe original

# Ciframos las ausencias
def encrypt_absences_rsa(absences):                        # Definimos la función para cifrar las ausencias
    absences = absences.replace("[", "").replace("]", "").replace("-", "").replace(" ", "")  # Eliminamos corchetes, guiones y espacios
    encrypted_absences = [rsa_encrypt(int(absence), n, e) for absence in absences.split(',')]  # Ciframos cada ausencia individualmente
    return encrypted_absences                              # Devolvemos las ausencias cifradas

df_workers_rsa['absences'] = df_workers_rsa['absences'].apply(encrypt_absences_rsa)  # Aplicamos el cifrado RSA a la columna de ausencias

# Mostramos el dataframe con ausencias cifradas
df_workers_rsa.head()                                      # Mostramos las primeras filas del dataframe cifrado con RSA


,id,name,sex,age,zip_code,num_absences,absences,insurance
0,1,Dwain Hathaway,M,27,8507,23,"[13548500888, 5103837441, 11430239684, 7964062...",1
1,2,Garnett Wolf,M,28,28009,6,"[5103837441, 5528809277, 13765257715, 25924984...",1
2,3,Roy Horsfall,M,29,39011,113,"[11000590288, 9074181250, 11000590288, 1100059...",0
3,4,Kory Easton,M,30,28036,1,[2993075967],0
4,5,Noel Spurling,F,31,49326,19,"[7840850750, 12452260183, 10766506889, 1245226...",0


**5.2** — Encrypt the absences with ElGamal using the given public key (`df_workers_elgamal`).

In [22]:
public_key_elgamal = (2481101477, 747698648, 249871017) # (p, alpha, beta)

In [23]:
from Crypto.PublicKey import ElGamal                       # Importamos la librería para manejar claves ElGamal
from Crypto.Random import random                           # Importamos la librería para generación de números aleatorios
from Crypto.Util.number import bytes_to_long, long_to_bytes  # Importamos utilidades para convertir entre bytes y números largos

# Clave pública ElGamal
p, alpha, beta = public_key_elgamal                        # Desempaquetamos los valores de la clave pública

def elgamal_encrypt(message, p, alpha, beta):              # Definimos la función para cifrar un mensaje usando ElGamal
    """Cifra un mensaje usando ElGamal"""
    k = random.StrongRandom().randint(1, p-2)              # Generamos un número aleatorio k
    gamma = pow(alpha, k, p)                               # Calculamos gamma = alpha^k mod p
    delta = (message * pow(beta, k, p)) % p                # Calculamos delta = mensaje * beta^k mod p
    return (gamma, delta)                                  # Devolvemos la pareja (gamma, delta)

# Creamos copia del dataframe para cifrado ElGamal
df_workers_elgamal = df_workers.copy()                     # Creamos una copia del dataframe original

# Ciframos las ausencias
def encrypt_absences_elgamal(absences):                    # Definimos la función para cifrar las ausencias
    absences = absences.replace("[", "").replace("]", "").replace("-", "").replace(" ", "")  # Eliminamos corchetes, guiones y espacios
    encrypted_absences = [elgamal_encrypt(int(absence), p, alpha, beta) for absence in absences.split(',')]  # Ciframos cada ausencia individualmente
    return encrypted_absences                              # Devolvemos las ausencias cifradas

df_workers_elgamal['absences'] = df_workers_elgamal['absences'].apply(encrypt_absences_elgamal)  # Aplicamos el cifrado ElGamal a la columna de ausencias

# Mostramos el dataframe con ausencias cifradas
df_workers_elgamal.head()                                  # Mostramos las primeras filas del dataframe cifrado con ElGamal


,id,name,sex,age,zip_code,num_absences,absences,insurance
0,1,Dwain Hathaway,M,27,8507,23,"[(1275726465, 176980684), (10317241, 809079895...",1
1,2,Garnett Wolf,M,28,28009,6,"[(1595876677, 482603945), (367366088, 18609196...",1
2,3,Roy Horsfall,M,29,39011,113,"[(2257279928, 129977756), (2338507814, 1803158...",0
3,4,Kory Easton,M,30,28036,1,"[(339126772, 261528839)]",0
4,5,Noel Spurling,F,31,49326,19,"[(2401565295, 455947838), (1052050456, 2149880...",0


**5.3** — Encrypt the absences with the A5/1 stream cipher initialized from the given register states (`df_workers_a5`).

In [24]:
initial_state_0 = [0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1]
initial_state_1 = [1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1]
initial_state_2 = [1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1]

In [25]:
class A5_1:                                                # Definimos la clase A5/1 para cifrado de flujo
    def __init__(self, initial_state_0, initial_state_1, initial_state_2):
        # Inicializar registros con los estados iniciales proporcionados
        self.R1 = initial_state_0                          # Registro R1 con el estado inicial
        self.R2 = initial_state_1                          # Registro R2 con el estado inicial
        self.R3 = initial_state_2                          # Registro R3 con el estado inicial

    def majority(self, x, y, z):                           # Calcula el bit de mayoría entre x, y y z
        return (x & y) | (x & z) | (y & z)

    def step(self):                                        # Realiza un paso del algoritmo A5/1
        maj = self.majority(self.R1[8], self.R2[10], self.R3[10])  # Calcula el bit de mayoría
        if self.R1[8] == maj:                              # Si el bit de mayoría coincide con R1[8], se desplaza R1
            self.R1 = [self.R1[18]] + self.R1[:18]
        if self.R2[10] == maj:                             # Si el bit de mayoría coincide con R2[10], se desplaza R2
            self.R2 = [self.R2[21]] + self.R2[:21]
        if self.R3[10] == maj:                             # Si el bit de mayoría coincide con R3[10], se desplaza R3
            self.R3 = [self.R3[22]] + self.R3[:22]

    def get_key_stream(self, length):                      # Genera una secuencia de claves de longitud dada
        key_stream = []                                    # Inicializa la secuencia de claves
        for _ in range(length):                            # Repite hasta generar la longitud requerida
            self.step()                                    # Realiza un paso del algoritmo
            key_bit = self.R1[-1] ^ self.R2[-1] ^ self.R3[-1]  # Calcula el bit de clave como XOR de los últimos bits de los registros
            key_stream.append(key_bit)                     # Añade el bit de clave a la secuencia
        return key_stream                                  # Devuelve la secuencia de claves

    def encrypt(self, message):                            # Cifra un mensaje usando A5/1
        key_stream = self.get_key_stream(len(message))     # Genera una secuencia de claves de la misma longitud que el mensaje
        encrypted_message = [m ^ k for m, k in zip(message, key_stream)]  # Cifra cada bit del mensaje con la secuencia de claves
        return encrypted_message                           # Devuelve el mensaje cifrado

# Creamos copia del dataframe para cifrado A5/1
df_workers_a5 = df_workers.copy()                          # Creamos una copia del dataframe original

# Ciframos las ausencias
def encrypt_absences_a5(absences):                         # Definimos la función para cifrar las ausencias
    a5 = A5_1(initial_state_0, initial_state_1, initial_state_2)  # Inicializamos la clase A5/1 con los estados iniciales
    absences = absences.replace("[", "").replace("]", "").replace("-", "").replace(" ", "")  # Eliminamos corchetes, guiones y espacios
    absences_bits = [int(bit) for absence in absences.split(',') for bit in bin(int(absence))[2:].zfill(24)]  # Convertimos las ausencias en bits
    encrypted_absences_bits = a5.encrypt(absences_bits)    # Ciframos los bits de las ausencias usando A5/1
    encrypted_absences = ''.join(str(bit) for bit in encrypted_absences_bits)  # Convertimos los bits cifrados en una cadena
    return encrypted_absences                              # Devolvemos las ausencias cifradas

df_workers_a5['absences'] = df_workers_a5['absences'].apply(encrypt_absences_a5)  # Aplicamos el cifrado A5/1 a la columna de ausencias

# Mostramos el dataframe con ausencias cifradas
df_workers_a5.head()                                       # Mostramos las primeras filas del dataframe cifrado con A5/1

,id,name,sex,age,zip_code,num_absences,absences,insurance
0,1,Dwain Hathaway,M,27,8507,23,1111000001010101001000100100000000100111011001...,1
1,2,Garnett Wolf,M,28,28009,6,1110000110001110110101000101101011101111110101...,1
2,3,Roy Horsfall,M,29,39011,113,1000101110110011000000111000111100101100010010...,0
3,4,Kory Easton,M,30,28036,1,1101100101110100000110101,0
4,5,Noel Spurling,F,31,49326,19,1010101000110110101000100001011110101111100111...,0


**Exercise 6:** What can an attacker holding only the ciphertext infer under each scheme? **6.1** — Can the number of absences per record be deduced?

In [26]:
# Respuesta RSA
# Ejemplo de lista cifrada con RSA
encrypted_absences_rsa = df_workers_rsa.iloc[0]['absences']  # Obtiene las ausencias cifradas del primer registro
num_absences_rsa = len(encrypted_absences_rsa)  # Cuenta el número de ausencias cifradas
num_absences_rsa  # Devuelve el número de ausencias

23

<div class="alert alert-block alert-success">
Justificación:

Para RSA, cada ausencia es cifrada individualmente. Por lo tanto, el número de ausencias puede deducirse directamente del número de elementos cifrados en la lista de ausencias.

En este caso, la longitud de la "lista encrypted_absences_rsa" indica directamente el número de ausencias. Este enfoque es posible porque el cifrado RSA no altera el número de elementos en la lista, solo cifra cada uno individualmente.


</div>

In [27]:
# Respuesta ElGamal
# Ejemplo de lista cifrada con ElGamal
encrypted_absences_elgamal = df_workers_elgamal.iloc[0]['absences']  # Obtiene las ausencias cifradas del primer registro
num_absences_elgamal = len(encrypted_absences_elgamal)  # Cuenta el número de ausencias cifradas
num_absences_elgamal  # Devuelve el número de ausencias

23

<div class="alert alert-block alert-success">
Justificación:

Al igual que RSA, ElGamal cifra cada ausencia individualmente. Por lo tanto, el número de ausencias puede deducirse del número de elementos cifrados en la lista de ausencias.


La longitud de la lista "encrypted_absences_elgamal" indica el número de ausencias. Este método es efectivo porque el cifrado ElGamal también cifra cada elemento individualmente sin cambiar la estructura de la lista.



</div>

In [28]:
# Respuesta A5/1
# Ejemplo de lista cifrada con A5/1
encrypted_absences_a5 = df_workers_a5.iloc[0]['absences']  # Obtiene las ausencias cifradas del primer registro
len(encrypted_absences_a5)  # Devuelve la longitud de la cadena cifrada

612

<div class="alert alert-block alert-success">
Justificación:

Para A5/1, el proceso de cifrado convierte toda la lista de ausencias en una cadena continua de bits. Esto hace que sea más difícil contar directamente las ausencias porque la estructura de los datos originales se pierde en el proceso de cifrado.

En este caso, no es posible contar directamente el número de ausencias sin conocer la longitud de los elementos originales, ya que todo se convierte en una cadena continua de bits.


</div>

**6.2** — Given two records, can we tell whether the two people were absent on the same occasion?

In [29]:
# Respuesta RSA

# Comparamos ausencias cifradas con RSA
absences1_rsa = set(df_workers_rsa.iloc[0]['absences'])  # Obtiene el conjunto de ausencias cifradas del primer registro
absences2_rsa = set(df_workers_rsa.iloc[1]['absences'])  # Obtiene el conjunto de ausencias cifradas del segundo registro
common_absences_rsa = absences1_rsa.intersection(absences2_rsa)  # Encuentra las ausencias comunes
common_absences_rsa  # Devuelve las ausencias comunes

{5103837441}

<div class="alert alert-block alert-success">
Justificación:

Para RSA, comparar ausencias entre diferentes registros puede realizarse comparando directamente los elementos cifrados.

Al ser determinista, cifra cada elemento individualmente, por lo que podemos comparar directamente las ausencias cifradas para encontrar coincidencias.


</div>

In [30]:
# Respuesta ElGamal

# Comparamos ausencias cifradas con ElGamal
absences1_elgamal = set(df_workers_elgamal.iloc[0]['absences'])  # Obtiene el conjunto de ausencias cifradas del primer registro
absences2_elgamal = set(df_workers_elgamal.iloc[1]['absences'])  # Obtiene el conjunto de ausencias cifradas del segundo registro
common_absences_elgamal = absences1_elgamal.intersection(absences2_elgamal)  # Encuentra las ausencias comunes
common_absences_elgamal  # Devuelve las ausencias comunes

set()

<div class="alert alert-block alert-success">
Justificación:

Para ElGamal, cada ausencia se cifra de manera diferente incluso si los valores originales son iguales, debido a la naturaleza probabilística del cifrado.

Dado que produce un cifrado diferente para el mismo valor original, no es posible encontrar coincidencias directamente comparando los elementos cifrados.


</div>

In [31]:
# Respuesta A5/1
# Comparamos ausencias cifradas con A5/1
absences1_a5 = df_workers_a5.iloc[0]['absences']  # Obtiene las ausencias cifradas del primer registro
absences2_a5 = df_workers_a5.iloc[1]['absences']  # Obtiene las ausencias cifradas del segundo registro

<div class="alert alert-block alert-success">
Justificación:

Para A5/1, comparar ausencias entre diferentes registros es extremadamente difícil debido a la naturaleza del cifrado de flujo, que convierte toda la lista en una cadena continua de bits.

A5/1 mezcla todos los bits de las ausencias, por lo que no es posible determinar si hay coincidencias de ausencias comparando directamente los datos cifrados.



</div>

**6.3** — Can any plaintext absence value be recovered from the encrypted dataset alone?

In [32]:
# Respuesta RSA
# Intentamos deducir ausencias usando un ataque de diccionario
# Supongamos que conocemos la clave pública
n, e = public_key_rsa

# Función para intentar descifrar ausencias
def try_decrypt_rsa(encrypted_absences):
    possible_absences = []
    for encrypted in encrypted_absences:
        for absence in range(0, 1000000):  # Asumimos que el valor de las ausencias está en este rango
            if pow(absence, e, n) == encrypted:
                possible_absences.append(absence)
                break
    return possible_absences

# Probamos con un registro
encrypted_absences_rsa = df_workers_rsa.iloc[0]['absences']  # Obtiene las ausencias cifradas del primer registro
decrypted_absences_rsa = try_decrypt_rsa(encrypted_absences_rsa)  # Intenta descifrar las ausencias
decrypted_absences_rsa  # Devuelve las ausencias descifradas

[]

<div class="alert alert-block alert-success">
Justificación:

Para RSA, si conocemos el texto cifrado y la clave pública, podemos intentar un ataque de diccionario para deducir las ausencias.

Si los valores de las ausencias están en un rango limitado, podemos intentar un ataque de diccionario para deducir los valores originales.


</div>

In [33]:
# Respuesta ElGamal
# No se puede realizar un ataque de diccionario con ElGamal debido a la naturaleza probabilística del cifrado.



<div class="alert alert-block alert-success">
Justificación:

Para ElGamal, debido a su naturaleza probabilística, un ataque de diccionario no es factible sin la clave privada.


</div>

In [34]:
# Respuesta A5/1

# No se puede deducir las ausencias originales sin conocer la clave inicial para A5/1.


<div class="alert alert-block alert-success">
Justificación:

Para A5/1, sin conocer la clave inicial, no es posible deducir las ausencias laborales originales.



</div>

**Exercise 7:** Check whether the predicted binary attribute `insurance` satisfies the independence property with respect to the protected attribute `sex` (chi-squared test).

In [35]:
# Respuesta
from scipy.stats import chi2_contingency

# Creamos tabla de contingencia
contingency_table = pd.crosstab(df_workers['sex'], df_workers['insurance'])  # Crea una tabla de contingencia con los valores de 'sex' y 'insurance'
print("Tabla de Contingencia:")
print(contingency_table)  # Muestra la tabla de contingencia

# Resumen de la distribución de los datos
print("\nResumen de la Distribución de los Datos:")
print(df_workers['sex'].value_counts())  # Muestra la distribución de los datos para 'sex'
print(df_workers['insurance'].value_counts())  # Muestra la distribución de los datos para 'insurance'

# Realizamos la prueba de chi-cuadrado
chi2, p, _, _ = chi2_contingency(contingency_table)  # Realiza la prueba de chi-cuadrado sobre la tabla de contingencia
print("\nResultados de la Prueba de Chi-Cuadrado:")
print(f"Chi-cuadrado: {chi2}")  # Muestra el valor de chi-cuadrado
print(f"Valor p: {p}")  # Muestra el valor p

Tabla de Contingencia:
insurance  0  1
sex            
F          3  8
M          4  7

Resumen de la Distribución de los Datos:
sex
M    11
F    11
Name: count, dtype: int64
insurance
1    15
0     7
Name: count, dtype: int64

Resultados de la Prueba de Chi-Cuadrado:
Chi-cuadrado: 0.0
Valor p: 1.0


<div class="alert alert-block alert-success">
Justificación:

Hipótesis Nula (H0): no hay relación entre el sexo y la concesión del seguro (son independientes).

Hipótesis Alternativa (H1): hay una relación entre el sexo y la concesión del seguro (no son independientes).

Dado que el valor p (1.0) es mucho mayor que 0.05, no rechazamos la hipótesis nula. Esto sugiere que no hay evidencia suficiente para afirmar que existe una relación significativa entre el sexo y la concesión del seguro en este conjunto de datos. En otras palabras, la concesión del seguro parece ser independiente del sexo de la persona, cumpliendo así la propiedad de independencia respecto al atributo protegido sex.




</div>